In [2]:
# SETUP AND ENVIRONMENT
#I used Google Colab for this assignment, which provided a cloud-based Python environment with TensorFlow pre-installed. This eliminated the need for local installation and provided access to free GPU resources, which accelerated the training process. The setup was straightforward - I simply created a new notebook and started coding immediately.

# Cell 1: Import libraries and load data
import tensorflow as tf
from tensorflow.keras.datasets import mnist
import numpy as np

print("Step 1: Loading and preprocessing MNIST dataset...")
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0
x_train = x_train.reshape((-1, 28, 28, 1))
x_test = x_test.reshape((-1, 28, 28, 1))
print(f"Training samples: {len(x_train)}, Test samples: {len(x_test)}")

# Cell 2: Build and compile model
print("\nStep 2: Building the model...")
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

print("\nStep 3: Compiling the model...")
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

# Cell 3: Train the model
print("\nStep 4: Training the model (this will take a few minutes)...")
history = model.fit(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

# Cell 4: Evaluate
print("\nStep 5: Evaluating the model...")
test_loss, test_accuracy = model.evaluate(x_test, y_test)
print(f"\n{'='*50}")
print(f"FINAL RESULTS:")
print(f"Test accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"Test loss: {test_loss:.4f}")
print(f"{'='*50}")

# Cell 5: Convert to TFLite
print("\nStep 6: Converting model to TFLite format...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save the model
with open('model.tflite', 'wb') as f:
    f.write(tflite_model)

import os
tflite_size = os.path.getsize('model.tflite') / 1024  # Size in KB
print(f"Model saved as 'model.tflite' ({tflite_size:.2f} KB)")

# Cell 6: Test TFLite model
print("\nStep 7: Testing TFLite model inference...")
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Test with 5 samples
print("\nTesting 5 sample predictions:")
print(f"{'Sample':<10} {'Predicted':<12} {'Actual':<10} {'Correct?':<10}")
print("-" * 45)

correct = 0
for i in range(5):
    test_image = np.expand_dims(x_test[i], axis=0).astype(np.float32)
    interpreter.set_tensor(input_details[0]['index'], test_image)
    interpreter.invoke()
    output_data = interpreter.get_tensor(output_details[0]['index'])
    predicted_label = np.argmax(output_data)
    actual_label = y_test[i]
    is_correct = predicted_label == actual_label
    if is_correct:
        correct += 1
    print(f"{i+1:<10} {predicted_label:<12} {actual_label:<10} {'✓' if is_correct else '✗':<10}")

print(f"\nSample accuracy: {correct}/5 ({correct/5*100:.0f}%)")
print("\n✅ Deployment simulation complete!")

# Cell 7: Download the model (optional)
from google.colab import files
files.download('model.tflite')
print("TFLite model ready for download!")

Step 1: Loading and preprocessing MNIST dataset...
11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training samples: 60000, Test samples: 10000

Step 2: Building the model...

Step 3: Compiling the model...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 5408)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       692,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 693,962 (2.65 MB)

...


Step 4: Training the model (this will take a few minutes)...
Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 51s 26ms/step - accuracy: 0.9126 - loss: 0.2948 - val_accuracy: 0.9782 - val_loss: 0.0668
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 49s 26ms/step - accuracy: 0.9835 - loss: 0.0536 - val_accuracy: 0.9844 - val_loss: 0.0444
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 82s 26ms/step - accuracy: 0.9898 - loss: 0.0321 - val_accuracy: 0.9872 - val_loss: 0.0394
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 48s 26ms/step - accuracy: 0.9928 - loss: 0.0219 - val_accuracy: 0.9861 - val_loss: 0.0442
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 48s 26ms/step - accuracy: 0.9966 - loss: 0.0124 - val_accuracy: 0.9863 - val_loss: 0.0446

Step 5: Evaluating the model...
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9816 - loss: 0.0571

FINAL RESULTS:
Test accuracy: 0.9863 (98.63%)
Test loss: 0.0446

Step 6: Converting model to TFLite format...
Saved artifact at '/tmp/tmpq5efwp6m'. The following endpoints ar

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

TFLite model ready for download!
